# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/palkinsuneja/palkin-flyrank-ml-internship-july-to-sept-2026/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip install -q duckdb huggingface_hub pandas numpy

In [18]:
import os
import duckdb
from google.colab import userdata

# Get token securely from Colab Secrets
hf_token = userdata.get("HF_TOKEN")

# Connect to DuckDB
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# Authenticate DuckDB with Hugging Face
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{hf_token}')"
)

# Warehouse paths
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

# Feature and label windows
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

# Dimension tables
DIM = f"read_parquet('{REL}/dim_content.parquet')"
CLI = f"read_parquet('{REL}/dim_clients.parquet')"

print("Connected successfully.")
print("Feature window: February 2026")
print("Label window: March 2026")

Connected successfully.
Feature window: February 2026
Label window: March 2026


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Fields: feature / label / context / excluded

**Features:** February 2026 Search Console and Analytics signals that are knowable at decision time, including `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_pageviews`, and `ga4_sessions`.

**Label:** The March 2026 outcome used to identify content that may need refresh attention. The label is based on later performance, so it is not available when the February decision is made.

**Context:** `client_hash_id`, `content_hash_id`, `report_date`, and data-availability fields provide the identifiers, time context, and coverage information needed to interpret the performance signals.

**Excluded:** Label-derived fields such as `trend_direction` or `trend_pct` are deliberately excluded from the feature set because they use future outcome information and would cause target leakage.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Inspect the available fields in the warehouse fact table
schema_df = con.execute(f"""

DESCRIBE SELECT *

FROM {FEB}

LIMIT 1

""").df()

schema_df[["column_name", "column_type"]]

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


In [20]:
from huggingface_hub import hf_hub_download

try:
    test_file = hf_hub_download(
        repo_id="FlyRank/internship-warehouse",
        filename="fact_content_daily_performance/month=2026-02/data_0.parquet",
        repo_type="dataset",
        token=hf_token
    )

    print("Parquet file access confirmed.")
    print("Downloaded successfully.")

except Exception as e:
    print("Parquet file access failed:")
    print(type(e).__name__)
    print(str(e)[:500])

Parquet file access confirmed.
Downloaded successfully.


In [21]:
print("Token loaded:", bool(hf_token))
print("Token length:", len(hf_token) if hf_token else 0)

Token loaded: True
Token length: 37


In [22]:
from huggingface_hub import HfApi

api = HfApi(token=hf_token)

try:
    info = api.dataset_info("FlyRank/internship-warehouse")
    print("Dataset access confirmed.")
    print("Dataset:", info.id)
except Exception as e:
    print("Dataset access check failed:")
    print(type(e).__name__)
    print(str(e)[:500])

Dataset access confirmed.
Dataset: FlyRank/internship-warehouse


In [23]:
from huggingface_hub import hf_hub_download

try:
    test_file = hf_hub_download(
        repo_id="FlyRank/internship-warehouse",
        filename="fact_content_daily_performance/month=2026-02/data_0.parquet",
        repo_type="dataset",
        token=hf_token
    )

    print("Parquet file access confirmed.")
    print("Downloaded successfully.")

except Exception as e:
    print("Parquet file access failed:")
    print(type(e).__name__)
    print(str(e)[:500])

Parquet file access confirmed.
Downloaded successfully.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Leakage trap

To demonstrate target leakage, I temporarily add a label-derived field, `trend_pct`, to the feature set. Because this field is derived from outcome information, it is not available at the February decision moment. A

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3 — Verify the data contract with three queries

# Query 1 — Grain:
# After aggregating daily records, one decision row should represent
# one unique client-content pair.
grain_check = con.execute(f"""
SELECT
    COUNT(*) AS decision_rows,
    COUNT(DISTINCT client_hash_id || '|' || content_hash_id)
        AS distinct_client_content_pairs
FROM (
    SELECT
        client_hash_id,
        content_hash_id
    FROM {FEB}
    GROUP BY client_hash_id, content_hash_id
)
""").df()

print("Query 1 — Grain check")
display(grain_check)


# Query 2 — February slice row count and date span
slice_check = con.execute(f"""
SELECT
    COUNT(*) AS daily_rows,
    COUNT(DISTINCT client_hash_id || '|' || content_hash_id)
        AS distinct_client_content_pairs,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {FEB}
""").df()

print("Query 2 — February slice count and date span")
display(slice_check)


# Query 3 — GSC availability
# Explicit IS TRUE check as required by the assignment.
availability_check = con.execute(f"""
SELECT
    COUNT(*) AS rows_with_gsc_available
FROM {FEB}
WHERE gsc_data_available IS TRUE
""").df()

print("Query 3 — GSC availability")
display(availability_check)

Query 1 — Grain check


,decision_rows,distinct_client_content_pairs
0,321546,321546


Query 2 — February slice count and date span


,daily_rows,distinct_client_content_pairs,first_date,last_date
0,7355108,321546,2026-02-01,2026-02-28


Query 3 — GSC availability


,rows_with_gsc_available
0,2621783


In [25]:
# Five-feature frame for the February decision window

feature_frame = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(ga4_pageviews) AS ga4_pageviews,
    SUM(ga4_sessions) AS ga4_sessions
FROM {FEB}
GROUP BY client_hash_id, content_hash_id
LIMIT 10
""").df()

print("Five-feature frame — February 2026")
display(feature_frame)

Five-feature frame — February 2026


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions
0,client_3ffa76342f366962,content_fb84747a57b8b665,0.0,0.0,NaN,NaN,NaN
1,client_3ffa76342f366962,content_feccf822ac21326e,0.0,0.0,NaN,NaN,NaN
2,client_3ffa76342f366962,content_17cf93c10413ebe9,0.0,0.0,NaN,NaN,NaN
3,client_3ffa76342f366962,content_a9905735266f8697,0.0,0.0,NaN,NaN,NaN
4,client_3ffa76342f366962,content_31c34765e7bba2f0,0.0,0.0,NaN,NaN,NaN
5,client_3ffa76342f366962,content_dcdf7e842dd640a5,0.0,0.0,NaN,NaN,NaN
6,client_3ffa76342f366962,content_dcf77b02e9c13344,0.0,0.0,NaN,NaN,NaN
7,client_3ffa76342f366962,content_3c3439c4de063402,0.0,0.0,NaN,NaN,NaN
8,client_3ffa76342f366962,content_14aa55a733a83e24,0.0,0.0,NaN,NaN,NaN
9,client_3ffa76342f366962,content_1ffdc7d0f4c3c5b8,0.0,0.0,NaN,NaN,NaN


### When each feature is available

- **gsc_impressions** — Knowable at the decision moment because it is aggregated from Search Console data in the February 2026 feature window.
- **gsc_clicks** — Knowable at the decision moment because it is aggregated from Search Console data available during the February 2026 feature window.
- **gsc_avg_position** — Knowable at the decision moment because it is calculated from Search Console position data observed during the February 2026 feature window.
- **ga4_pageviews** — Knowable at the decision moment because it is aggregated from Analytics data available during the February 2026 feature window.
- **ga4_sessions** — Knowable at the decision moment because it is aggregated from Analytics session data available during the February 2026 feature window.

In [26]:
# Deliberate leakage experiment
# March outcome is intentionally used as a feature to demonstrate leakage.

leaky_frame = con.execute(f"""
SELECT
    f.client_hash_id,
    f.content_hash_id,
    SUM(f.gsc_clicks) AS feb_clicks,
    SUM(f.gsc_impressions) AS feb_impressions,
    m.march_clicks AS leaked_march_clicks
FROM {FEB} f
JOIN (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS march_clicks
    FROM {MAR}
    GROUP BY client_hash_id, content_hash_id
) m
ON f.client_hash_id = m.client_hash_id
AND f.content_hash_id = m.content_hash_id
GROUP BY
    f.client_hash_id,
    f.content_hash_id,
    m.march_clicks
LIMIT 1000
""").df()

print("Leaky experiment frame created.")
print("Rows:", len(leaky_frame))
display(leaky_frame.head())

Leaky experiment frame created.
Rows: 1000


,client_hash_id,content_hash_id,feb_clicks,feb_impressions,leaked_march_clicks
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,7.0,4270.0,7.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2.0,440.0,0.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,4.0,5271.0,6.0
3,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,1.0,260.0,1.0
4,client_73cda7b4e4f265ea,content_1f380a642aed423b,0.0,282.0,1.0


In [27]:
# Show how a label-derived feature can make the score look artificially strong

from sklearn.metrics import accuracy_score

# March outcome: 1 if the content received zero clicks in March
leaky_eval = leaky_frame.copy()
leaky_eval["label"] = (leaky_eval["leaked_march_clicks"] == 0).astype(int)

# Directly using the leaked outcome as a prediction signal
leaky_eval["leaky_prediction"] = (
    leaky_eval["leaked_march_clicks"] == 0
).astype(int)

leaky_score = accuracy_score(
    leaky_eval["label"],
    leaky_eval["leaky_prediction"]
)

print(f"Leaky score: {leaky_score:.3f}")
print("The score is artificially perfect because the feature contains the future outcome.")

Leaky score: 1.000
The score is artificially perfect because the feature contains the future outcome.


In [28]:
# Remove the leaked outcome-derived column before modeling

honest_feature_frame = feature_frame.copy()

print("Leaked column removed.")
print("Honest feature columns:")
print(list(honest_feature_frame.columns))

Leaked column removed.
Honest feature columns:
['client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

This slice does not provide a complete view of every content item's performance. GSC availability varies across rows, so content without GSC data cannot be evaluated using the same search-performance signals.

The February feature window and March outcome window cover only adjacent months, so the contract does not capture longer-term seasonality or changes in search behavior.

Therefore, the resulting ranking should be treated as decision support for refresh prioritization, not as a complete or causal explanation of content performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Self-check

- [x] Every section above is filled — contract answers, verification queries, five-feature frame, leakage experiment, and data limitation.
- [x] The three verification queries were executed successfully, including the GSC availability check using `IS TRUE`.
- [x] The five features are based only on February 2026 information available at the decision moment.
- [x] A deliberate leakage experiment was performed using March 2026 outcome information and produced an artificially perfect score of 1.000.
- [x] The leaked outcome-derived column was removed before modeling, leaving only honest feature columns.
- [x] No client names, private URLs, or private search queries are included.
- [x] Claims are framed as observed data checks and decision support, not causal conclusions.
- [x] The notebook is ready to be saved under `work/notebooks/w03_data_contract.ipynb` and committed to the repository.